In [0]:
-- =====================================================================
-- LBE Shareables & Desserts Scene+ Points Offer — Dec 2026 projection
-- Scope: Rec Room + Playdium (BR-012), Restaurant% locations
-- Qualifying txn = net-positive qty of any offer item (BR-014)
-- Scanned txn    = any line SCENEFLAG = 'Yes' (BR-015)
-- Dates          = VisitDateId (TL-04, behavioural)
-- Projection     = Dec 2025 x (Jul+Aug 2026 / Jul+Aug 2025), per brand
-- =====================================================================

-- 0. Offer item list (stakeholder-provided, 100 IDs)
CREATE OR REPLACE TEMP VIEW offer_items AS
SELECT explode(array(
  '21010','21019','21045','22005','22024','22056','22069','22091','22183','22192',
  '23001','23007','25013','28000','28002','28027','28034','28035','28039','28042',
  '28049','28066','28071','28080','28087','28091','28092','28093','28094','28095',
  '28221','28281','28412','28423','28424','28427','28428','28430','28431','28949',
  '29320','29324','33059','33098','33107','33126','33136','40219','40220','40222',
  '40223','40224','40226','40230','40238','40239','40423','40424','40456','40573',
  '40598','40599','40799','250131','250132','250133','250134','250141','250152','250180',
  '250779','250899','260110','260111','260112','260113','260143','260144','260145','260146',
  '260147','260157','260158','260159','260212','260213','260214','260215','260220','260374',
  '260375','260376','260748','261093','261094','261095','261096','261097','261102','382611'
)) AS item_id;


-- 1. Transaction-level base: Jul-Aug 2025, Dec 2025, Jul-Aug 2026
CREATE OR REPLACE TEMP VIEW lbe_txn_base AS
WITH lbe_lines AS (
  SELECT t.LocationId, t.VisitDateId, t.Transaction_Number,
         SUBSTR(CAST(t.VisitDateId AS STRING), 1, 6) AS month,
         l.LC_Trade_Brand_Description AS brand,
         t.CDE_ID, t.SCENEFLAG,
         CASE WHEN o.item_id IS NOT NULL THEN t.Quantity ELSE 0 END AS offer_qty
  FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
  JOIN cpx_dataanalytics_gold.customer360.cpx_d_location l
    ON t.LocationId = l.LocationID
  LEFT JOIN cpx_dataanalytics_gold.customer360.cpx_d_item i
    ON t.ItemSkey = i.itemskey
  LEFT JOIN offer_items o
    ON CAST(i.Item_ID AS STRING) = o.item_id
  WHERE (   t.VisitDateId BETWEEN 20250701 AND 20250831
         OR t.VisitDateId BETWEEN 20251201 AND 20251231
         OR t.VisitDateId BETWEEN 20260701 AND 20260831)
    AND l.LC_Location_Type_Description LIKE 'Restaurant%'
    AND l.LC_Trade_Brand_Description IN ('Rec Room', 'Playdium')
)
SELECT brand, month, LocationId, VisitDateId, Transaction_Number,
       MAX(CASE WHEN SCENEFLAG = 'Yes' THEN 1 ELSE 0 END) AS is_scanned,
       COUNT(DISTINCT CDE_ID)                             AS n_cde_ids,
       SUM(offer_qty)                                     AS offer_net_qty
FROM lbe_lines
GROUP BY brand, month, LocationId, VisitDateId, Transaction_Number;


-- 2. Dec 2026 projection by brand
WITH monthly AS (
  SELECT brand, month,
         COUNT(*)        AS qualifying_txns,
         SUM(is_scanned) AS scanned_txns
  FROM lbe_txn_base
  WHERE offer_net_qty > 0
  GROUP BY brand, month
),
growth AS (
  SELECT brand,
         SUM(CASE WHEN month IN ('202607','202608') THEN qualifying_txns END)
       / SUM(CASE WHEN month IN ('202507','202508') THEN qualifying_txns END) AS g_qualifying,
         SUM(CASE WHEN month IN ('202607','202608') THEN scanned_txns END)
       / SUM(CASE WHEN month IN ('202507','202508') THEN scanned_txns END)    AS g_scanned
  FROM monthly
  GROUP BY brand
)
SELECT m.brand,
       m.qualifying_txns                          AS dec25_qualifying,
       ROUND(g.g_qualifying, 3)                   AS g_qualifying,
       ROUND(m.qualifying_txns * g.g_qualifying)  AS dec26_qualifying,
       m.scanned_txns                             AS dec25_scanned,
       ROUND(g.g_scanned, 3)                      AS g_scanned,
       ROUND(m.scanned_txns * g.g_scanned)        AS dec26_scanned,
       ROUND(100.0 * m.scanned_txns * g.g_scanned
                   / (m.qualifying_txns * g.g_qualifying), 1) AS dec26_scan_rate_pct
FROM monthly m
JOIN growth g ON m.brand = g.brand
WHERE m.month = '202512'
ORDER BY m.brand;


-- 3. Avg Scene+ points earned per scanned qualifying txn (Dec 2025)
SELECT COALESCE(t.brand, 'Total')  AS brand,
       ROUND(AVG(p.pts), 1)        AS avg_points_earned_per_scanned_txn
FROM lbe_txn_base t
JOIN (
  SELECT tr.LocationId, tr.VisitDateId, tr.Transaction_Number,
         SUM(tr.SCENE_Points_Earned) AS pts
  FROM cpx_dataanalytics_gold.customer360.c360_f_transaction tr
  WHERE tr.VisitDateId BETWEEN 20251201 AND 20251231
  GROUP BY 1, 2, 3
) p
  ON  t.LocationId = p.LocationId
  AND t.VisitDateId = p.VisitDateId
  AND t.Transaction_Number = p.Transaction_Number
WHERE t.month = '202512'
  AND t.offer_net_qty > 0
  AND t.is_scanned = 1
GROUP BY GROUPING SETS ((t.brand), ())
ORDER BY brand;


-- =====================================================================
-- CHECKS
-- =====================================================================

-- C1. Monthly inputs. Dec 2025 must match baseline:
--     Rec Room 14,815 / 3,773 | Playdium 350 / 42
SELECT brand, month,
       COUNT(*)                                     AS qualifying_txns,
       SUM(is_scanned)                              AS scanned_txns,
       ROUND(100.0 * SUM(is_scanned) / COUNT(*), 1) AS scan_rate_pct
FROM lbe_txn_base
WHERE offer_net_qty > 0
GROUP BY brand, month
ORDER BY brand, month;


-- C2. Transaction key unique across all months (want: no rows)
SELECT month, COUNT(*) AS txns_multi_cde
FROM lbe_txn_base
WHERE n_cde_ids > 1
GROUP BY month;


-- C3. Menu effect: offer items sold per brand and month
SELECT l.LC_Trade_Brand_Description                 AS brand,
       SUBSTR(CAST(t.VisitDateId AS STRING), 1, 6)  AS month,
       COUNT(DISTINCT o.item_id)                    AS offer_items_sold,
       SUM(t.Quantity)                              AS offer_units
FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
JOIN cpx_dataanalytics_gold.customer360.cpx_d_location l
  ON t.LocationId = l.LocationID
JOIN cpx_dataanalytics_gold.customer360.cpx_d_item i
  ON t.ItemSkey = i.itemskey
JOIN offer_items o
  ON CAST(i.Item_ID AS STRING) = o.item_id
WHERE (   t.VisitDateId BETWEEN 20250701 AND 20250831
       OR t.VisitDateId BETWEEN 20251201 AND 20251231
       OR t.VisitDateId BETWEEN 20260701 AND 20260831)
  AND l.LC_Location_Type_Description LIKE 'Restaurant%'
  AND l.LC_Trade_Brand_Description IN ('Rec Room', 'Playdium')
GROUP BY 1, 2
ORDER BY 1, 2;

-- Playdium scan rate by venue and month
SELECT b.month,
       l.Location_Name,
       COUNT(*)                                        AS qualifying_txns,
       SUM(b.is_scanned)                               AS scanned_txns,
       ROUND(100.0 * SUM(b.is_scanned) / COUNT(*), 1)  AS scan_rate_pct
FROM lbe_txn_base b
JOIN cpx_dataanalytics_gold.customer360.cpx_d_location l
  ON b.LocationId = l.LocationID
WHERE b.brand = 'Playdium'
  AND b.offer_net_qty > 0
GROUP BY b.month, l.Location_Name
ORDER BY l.Location_Name, b.month;

-- Item-level average selling price, Jul-Aug 2026
SELECT l.LC_Trade_Brand_Description                           AS brand,
       o.item_id,
       MAX(i.Item_Description)                                AS item_description,
       MAX(i.Item_Class_Description1)                         AS item_class,
       SUM(t.Quantity)                                        AS net_units,
       SUM(t.Net_Revenue)                                     AS net_revenue,
       ROUND(SUM(t.Net_Revenue) / NULLIF(SUM(t.Quantity), 0), 2) AS avg_price,
       SUM(CASE WHEN t.Net_Revenue = 0 THEN t.Quantity ELSE 0 END) AS zero_price_units
FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
JOIN cpx_dataanalytics_gold.customer360.cpx_d_location l
  ON t.LocationId = l.LocationID
JOIN cpx_dataanalytics_gold.customer360.cpx_d_item i
  ON t.ItemSkey = i.itemskey
JOIN offer_items o
  ON CAST(i.Item_ID AS STRING) = o.item_id
WHERE t.VisitDateId BETWEEN 20260701 AND 20260831
  AND l.LC_Location_Type_Description LIKE 'Restaurant%'
  AND l.LC_Trade_Brand_Description IN ('Rec Room', 'Playdium')
GROUP BY l.LC_Trade_Brand_Description, o.item_id
ORDER BY brand, net_revenue DESC;

-- Category share of offer items per brand and month
SELECT brand, month, category, offer_units, net_revenue,
       ROUND(100.0 * offer_units / SUM(offer_units) OVER (PARTITION BY brand, month), 1) AS units_share_pct,
       ROUND(100.0 * net_revenue / SUM(net_revenue) OVER (PARTITION BY brand, month), 1) AS revenue_share_pct
FROM (
  SELECT l.LC_Trade_Brand_Description                 AS brand,
         SUBSTR(CAST(t.VisitDateId AS STRING), 1, 6)  AS month,
         i.Item_Class_Description1                    AS category,
         SUM(t.Quantity)                              AS offer_units,
         SUM(t.Net_Revenue)                           AS net_revenue
  FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
  JOIN cpx_dataanalytics_gold.customer360.cpx_d_location l
    ON t.LocationId = l.LocationID
  JOIN cpx_dataanalytics_gold.customer360.cpx_d_item i
    ON t.ItemSkey = i.itemskey
  JOIN offer_items o
    ON CAST(i.Item_ID AS STRING) = o.item_id
  WHERE (   t.VisitDateId BETWEEN 20250701 AND 20250831
         OR t.VisitDateId BETWEEN 20251201 AND 20251231
         OR t.VisitDateId BETWEEN 20260701 AND 20260831)
    AND l.LC_Location_Type_Description LIKE 'Restaurant%'
    AND l.LC_Trade_Brand_Description IN ('Rec Room', 'Playdium')
  GROUP BY 1, 2, 3
)
ORDER BY brand, month, units_share_pct DESC;

-- Scene share of offer items per brand, month and category
SELECT brand, month, category, offer_units, scene_units,
       ROUND(100.0 * scene_units / NULLIF(offer_units, 0), 1)                            AS scene_attach_pct,
       ROUND(100.0 * scene_units / SUM(scene_units) OVER (PARTITION BY brand, month), 1) AS scene_units_share_pct
FROM (
  SELECT b.brand,
         b.month,
         i.Item_Class_Description1                                   AS category,
         SUM(t.Quantity)                                             AS offer_units,
         SUM(CASE WHEN b.is_scanned = 1 THEN t.Quantity ELSE 0 END)  AS scene_units
  FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
  JOIN lbe_txn_base b
    ON  t.LocationId = b.LocationId
    AND t.VisitDateId = b.VisitDateId
    AND t.Transaction_Number = b.Transaction_Number
  JOIN cpx_dataanalytics_gold.customer360.cpx_d_item i
    ON t.ItemSkey = i.itemskey
  JOIN offer_items o
    ON CAST(i.Item_ID AS STRING) = o.item_id
  GROUP BY 1, 2, 3
)
ORDER BY brand, month, offer_units DESC;